In [ ]:
# Week 3 Day 4: Hyperparameter Tuning with GridSearchCV

## Today's Goal

Today I will learn how to systematically tune model hyperparameters using GridSreachCV while keeping the final test set independent.

## Learing Objective
- Understand why manual hyperparameter tuning becomes inefficient
- Understand the idea of a parameter grid
- Use GridSearchCV with cross-validation
- Combine GridSearchCV with a sklearn Pipeline
- Interpret best_params_ and best_score_
- Understand why the final test set should not be used during hyperparameter tuning

## Expected output

By the end of today, I should be able to:
    
1. Build a parameter grid for KNN
2. Run GridSearchCV using cross-validation
3. Explain how many model configurations are evaluated
4. Identify the best hyperparameters
5. Interpret the best cross-validation score
6. Evaluate the selected model once on the untouched test set

In [2]:
## Data Preparation
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Load Banknote Authentication dataset
banknote = fetch_openml(
    data_id=1462,
    as_frame=True
)

X = banknote.data
y = banknote.target

# Hold out the final test set
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Full dataset:", X.shape)
print("Development set:", X_dev.shape)
print("Final test set:", X_test.shape)

Full dataset: (1372, 4)
Development set: (1097, 4)
Final test set: (275, 4)


In [3]:
## KNN Pipeline and parameter Grid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Build preprocessing + model pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

# Define hyperparameters to search
param_grid = {
    "knn__n_neighbors":[3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"]
}

print(knn_pipeline)
print("\nParameter grid:")
print(param_grid)

Pipeline(steps=[('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])

Parameter grid:
{'knn__n_neighbors': [3, 5, 7, 9, 11], 'knn__weights': ['uniform', 'distance']}


In [4]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Define the cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

In [5]:
grid_search.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest mean CV accuracy:")
print(grid_search.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:
{'knn__n_neighbors': 3, 'knn__weights': 'uniform'}

Best mean CV accuracy:
0.9981818181818183


In [7]:
## Inspecting All Grid Search Results
import pandas as pd

# Convert GridSearchCV results into a DataFrame
results = pd.DataFrame(grid_search.cv_results_)

# Keep the columns we care about
results_table = results[
    [
        "param_knn__n_neighbors",
        "param_knn__weights",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

# Sort by GridSearchCV ranking
results_table = results_table.sort_values(
    by=["rank_test_score", "mean_test_score"],
    ascending=[True, False]
)

print(results_table.to_string(index=False))

 param_knn__n_neighbors param_knn__weights  mean_test_score  std_test_score  rank_test_score
                      3            uniform         0.998182        0.002227                1
                      3           distance         0.998182        0.002227                1
                      5            uniform         0.998182        0.002227                1
                      5           distance         0.998182        0.002227                1
                      7            uniform         0.998182        0.002227                1
                      7           distance         0.998182        0.002227                1
                      9            uniform         0.998182        0.002227                1
                      9           distance         0.998182        0.002227                1
                     11           distance         0.998182        0.002227                1
                     11            uniform         0.993620        0.0

In [10]:
## Final Test Evaluation
from sklearn.metrics import accuracy_score

# GridSearchCV already refitted the selected configuration
# on all development data
best_model = grid_search.best_estimator_

# Final prediction on the untouched test set
y_test_pred = best_model.predict(X_test)

final_test_accuracy = accuracy_score(y_test, y_test_pred)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest mean CV accuracy:")
print(grid_search.best_score_)

print("\nFinal test accuracy:")
print(final_test_accuracy)

Best parameters:
{'knn__n_neighbors': 3, 'knn__weights': 'uniform'}

Best mean CV accuracy:
0.9981818181818183

Final test accuracy:
1.0


In [ ]:
# Reflection

1. Why is GridSearchCV more reliable than choosing hyperparameters from one validation split?
Because GridSearchCV outputs mean CV accuracy and Standard Deviation which shows the model generalized performance and stability with 
avoiding the noise of a particular dataset.
2. Why does `best_params_` not necessarily mean that the returned configuration is clearly better than every other configuration?
Because if there are the same Mean CV accuracy and Std in different configurations, it just returns one selected configuration rather than 
all.
3. Why should the final test set remain untouched until GridSearchCV and model selection are finished?
Because the final test set should be independent without test set contamination and teh isolated test set shows 